# SimpleDet Showcase

This notebook is a compact, executable tour of the public SimpleDet API. It is intentionally self-contained: the cells build detector specs, compile native plans, inspect project-layout helpers, and report optional runtime registry availability without requiring a dataset download.

In [1]:
from importlib import metadata
import json

import simpledet
from simpledet import ProjectLayout, project_config_template
from simpledet.suite import (
    build_custom_encoder,
    build_detector,
    build_neck,
    compile_native_detector_plan,
    list_native_detector_families,
    list_native_head_families,
    list_native_neck_families,
)

try:
    package_version = metadata.version("simpledet")
except metadata.PackageNotFoundError:
    package_version = "source checkout"

print(f"SimpleDet package: {package_version}")
print(f"Imported from: {simpledet.__file__}")

SimpleDet package: 0.1.0
Imported from: /shared/home/rdelprete/PythonProjects/MMDET/simpledet/simpledet/__init__.py


## Dense and ROI detector families

The suite API keeps detector selection declarative while preserving the native family and head choices that the runtime needs.

In [2]:
def summarize_spec(architecture, *, in_channels=4, num_classes=3):
    spec = build_detector(
        architecture,
        encoder="resnet18.a1_in1k",
        neck=build_neck("FPN", out_channels=128, num_outs=5),
        num_classes=num_classes,
        in_channels=in_channels,
        pretrained=False,
    )
    plan = compile_native_detector_plan(spec)
    return {
        "architecture": plan.architecture,
        "family": plan.family,
        "encoder": plan.encoder.params["model_name"],
        "in_channels": plan.encoder.params["in_channels"],
        "neck": plan.neck.type,
        "head": plan.head.type if plan.head else "decoder-only",
        "classes": plan.num_classes,
    }

architectures = [
    "retinanet",
    "vfnet",
    "fovea",
    "reppoints",
    "yolof",
    "centernet",
    "faster_rcnn",
    "mask_rcnn",
    "grid_rcnn",
    "cascade_rcnn",
]
rows = [summarize_spec(name) for name in architectures]

print(f"{'architecture':<14} {'family':<7} {'head':<16} {'encoder':<18} {'bands':<5} {'classes':<7}")
print("-" * 76)
for row in rows:
    print(
        f"{row['architecture']:<14} {row['family']:<7} {row['head']:<16} "
        f"{row['encoder']:<18} {row['in_channels']:<5} {row['classes']:<7}"
    )

architecture   family  head             encoder            bands classes
----------------------------------------------------------------------------
retinanet      dense   RetinaHead       resnet18.a1_in1k   4     3      
vfnet          dense   VFNetHead        resnet18.a1_in1k   4     3      
fovea          dense   FoveaHead        resnet18.a1_in1k   4     3      
reppoints      dense   RepPointsHead    resnet18.a1_in1k   4     3      
yolof          dense   YOLOFHead        resnet18.a1_in1k   4     3      
centernet      dense   CenterNetHead    resnet18.a1_in1k   4     3      
faster_rcnn    roi     RetinaHead       resnet18.a1_in1k   4     3      
mask_rcnn      roi     RetinaHead       resnet18.a1_in1k   4     3      
grid_rcnn      roi     RetinaHead       resnet18.a1_in1k   4     3      
cascade_rcnn   roi     RetinaHead       resnet18.a1_in1k   4     3      


## Custom remote-sensing backbone plan

Custom encoders can declare their output channels up front. The compiled plan carries that shape contract forward so native backbones, necks, heads, and runtime settings stay explicit.

In [3]:
sentinel_encoder = build_custom_encoder(
    "SentinelConvNeXtTiny",
    feature_channels=(96, 192, 384, 768),
    imports=("my_project.backbones.sentinel",),
    in_channels=8,
    sensor="Sentinel-2",
)

sentinel_spec = build_detector(
    "vfnet",
    encoder=sentinel_encoder,
    neck=build_neck("FPN", out_channels=192, num_outs=4),
    num_classes=2,
    score_threshold=0.35,
)
sentinel_plan = compile_native_detector_plan(sentinel_spec)

payload = {
    "architecture": sentinel_plan.architecture,
    "family": sentinel_plan.family,
    "encoder": sentinel_plan.encoder.to_dict(),
    "neck": sentinel_plan.neck.to_dict(),
    "head": sentinel_plan.head.to_dict(),
    "overrides": sentinel_plan.overrides,
}
print(json.dumps(payload, indent=2))

{
  "architecture": "vfnet",
  "family": "dense",
  "encoder": {
    "kind": "encoder",
    "type": "SentinelConvNeXtTiny",
    "params": {
      "type": "SentinelConvNeXtTiny",
      "in_channels": 8,
      "sensor": "Sentinel-2",
      "feature_channels": [
        96,
        192,
        384,
        768
      ]
    },
    "imports": [
      "my_project.backbones.sentinel"
    ],
    "source": "config"
  },
  "neck": {
    "kind": "neck",
    "type": "FPN",
    "params": {
      "out_channels": 192,
      "num_outs": 4
    },
    "imports": [],
    "source": null
  },
  "head": {
    "kind": "head",
    "type": "VFNetHead",
    "params": {
      "num_classes": 2
    },
    "imports": [],
    "source": null
  },
  "overrides": {
    "score_threshold": 0.35
  }
}


## Project-layout helpers

The public API separates dataset layout, runtime settings, optimization, and detector definition. This keeps notebooks and automation pointed at the same operational contract.

In [4]:
layout = ProjectLayout("/data/sentinel-vessels", result_folder="/tmp/simpledet-runs")
report = layout.validation_report()

print(json.dumps(report["paths"], indent=2))
print("missing in this dry run:", [name for name, exists in report["exists"].items() if not exists])

template_preview = "\n".join(project_config_template("toml").splitlines()[:24])
print("\nTOML project template preview:\n")
print(template_preview)

{
  "dataset_root": "/data/sentinel-vessels",
  "images": "/data/sentinel-vessels/imgs",
  "annotations_dir": "/data/sentinel-vessels/Annotations",
  "train_annotations": "/data/sentinel-vessels/Annotations/train_annotations.json",
  "val_annotations": "/data/sentinel-vessels/Annotations/val_annotations.json",
  "test_annotations": "/data/sentinel-vessels/Annotations/test_annotations.json"
}
missing in this dry run: ['dataset_root', 'images', 'annotations_dir', 'train_annotations', 'val_annotations', 'test_annotations']

TOML project template preview:

[dataset]
data_root = "/path/to/dataset"
annot_file_train = "/path/to/dataset/Annotations/train_annotations.json"
annot_file_val = "/path/to/dataset/Annotations/val_annotations.json"
annot_file_test = "/path/to/dataset/Annotations/test_annotations.json"
data_prefix = "imgs/"
categories = ["wake"]
in_channels = 3
tif_channels_to_load = [1, 2, 3]

[runtime]
result_folder = "/tmp/simpledet-runs"
resize = 768
batch_size = 2
max_epochs = 12
s

## Optional native registry discovery

When the `simpledet[cpu]` runtime dependencies are installed, the same API can list native registered detector, head, and neck factories.

In [5]:
registry_views = {
    "detectors": list_native_detector_families(),
    "heads": list_native_head_families(),
    "necks": list_native_neck_families(),
}

for label, values in registry_views.items():
    preview = values[:10] if values else ["install simpledet[cpu] to load native runtime registries"]
    print(f"{label}: {preview}")

detectors: ['install simpledet[cpu] to load native runtime registries']
heads: ['install simpledet[cpu] to load native runtime registries']
necks: ['install simpledet[cpu] to load native runtime registries']


## What this demonstrates

This notebook exercises the package without external data: detector aliases, dense/ROI family routing, custom remote-sensing encoders, native plan compilation, project config generation, and optional runtime registry discovery.

In [6]:
summary = {
    "covered": [
        "dense detector aliases",
        "ROI detector aliases",
        "custom multi-band encoder specs",
        "native build-plan compilation",
        "project config templates",
        "optional runtime registry discovery",
    ],
    "next_runtime_commands": [
        "python -m pip install -e .[cpu]",
        "python -m simpledet --check-runtime",
        "python -m simpledet --list-detectors",
    ],
}
print(json.dumps(summary, indent=2))

{
  "covered": [
    "dense detector aliases",
    "ROI detector aliases",
    "custom multi-band encoder specs",
    "native build-plan compilation",
    "project config templates",
    "optional runtime registry discovery"
  ],
  "next_runtime_commands": [
    "python -m pip install -e .[cpu]",
    "python -m simpledet --check-runtime",
    "python -m simpledet --list-detectors"
  ]
}
